<a href="https://colab.research.google.com/github/cristiangromero/LGRT-PLP/blob/main/Unidad3/Interprete_UGD_PLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Intérprete Simple para el Lenguaje UGD-PLP
Este notebook implementa paso a paso un intérprete para un lenguaje de programación simple con expresiones aritméticas, asignaciones, condicionales `if`, bucles `while` y soporte parcial de tipos.



### 🔹 Paso 1: Tokenizador (Análisis Léxico)

### 🔹 Paso 2: Construcción del AST

In [6]:
import re

# ⚠️ Palabras clave ANTES de ID para evitar conflictos
TOKEN_REGEX = [
    ('NUMBER',    r'\d+'),
    ('IF',        r'\bif\b'),
    ('ELSE',      r'\belse\b'),
    ('WHILE',     r'\bwhile\b'),
    ('PRINT',     r'\bprint\b'),
    ('INT',       r'\bint\b'),
    ('FLOAT_T',   r'\bfloat\b'),
    ('STRING_T',  r'\bstring\b'),
    ('BOOL_T',    r'\bboolean\b'),
    ('STRING',    r'"[^"]*"'),           # Captura "texto"
    ('BOOL',      r'\btrue\b|\bfalse\b'), # Captura true o false
    ('ID',        r'[a-zA-Z_][a-zA-Z0-9_]*'),
    ('REL_OP',    r'==|!=|<=|>=|<|>'),
    ('OP_MUL',    r'[*/]'),
    ('OP_ADD',    r'[+\-]'),
    ('ASSIGN',    r'='),
    ('LPAREN',    r'\('),
    ('RPAREN',    r'\)'),
    ('LBRACE',    r'\{'),
    ('RBRACE',    r'\}'),
    ('SEMICOLON', r';'),
    ('SKIP',      r'[ \t\n]+'),
    ('MISMATCH',  r'.')

]

def tokenize(code):
    tokens = []
    pos = 0
    while pos < len(code):
        match = None
        for tok_type, regex in TOKEN_REGEX:
            pattern = re.compile(regex)
            match = pattern.match(code, pos)
            if match:
                if tok_type != 'SKIP':
                    tokens.append((tok_type, match.group()))
                pos = match.end()
                break
        if not match:
            raise SyntaxError(f"Caracter inválido: {code[pos]}")
    return tokens

# 🧪 Prueba del tokenizador
test_code = "x = 5; if (x > 0) { print(x); }"
print("Código de prueba:", test_code)
print("Tokens:", tokenize(test_code))

Código de prueba: x = 5; if (x > 0) { print(x); }
Tokens: [('ID', 'x'), ('ASSIGN', '='), ('NUMBER', '5'), ('SEMICOLON', ';'), ('IF', 'if'), ('LPAREN', '('), ('ID', 'x'), ('REL_OP', '>'), ('NUMBER', '0'), ('RPAREN', ')'), ('LBRACE', '{'), ('PRINT', 'print'), ('LPAREN', '('), ('ID', 'x'), ('RPAREN', ')'), ('SEMICOLON', ';'), ('RBRACE', '}')]


In [7]:
class ASTNode:
    pass

class Number(ASTNode):
    def __init__(self, value):
        self.value = int(value)

    def __repr__(self):
        return f"Number({self.value})"

class Var(ASTNode):
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return f"Var({self.name})"

class BinOp(ASTNode):
    def __init__(self, left, op, right):
        self.left = left
        self.op = op
        self.right = right

    def __repr__(self):
        return f"BinOp({self.left}, {self.op}, {self.right})"

class Assign(ASTNode):
    def __init__(self, var, expr):
        self.var = var
        self.expr = expr

    def __repr__(self):
        return f"Assign({self.var}, {self.expr})"

class Print(ASTNode):
    def __init__(self, expr):
        self.expr = expr

    def __repr__(self):
        return f"Print({self.expr})"

class If(ASTNode):
    def __init__(self, cond, then_branch, else_branch):
        self.cond = cond
        self.then_branch = then_branch
        self.else_branch = else_branch

    def __repr__(self):
        return f"If({self.cond}, {self.then_branch}, {self.else_branch})"

class While(ASTNode):
    def __init__(self, cond, body):
        self.cond = cond
        self.body = body

    def __repr__(self):
        return f"While({self.cond}, {self.body})"

class Block(ASTNode):
    def __init__(self, statements):
        self.statements = statements

    def __repr__(self):
        return f"Block({self.statements})"

class Declaration(ASTNode):
    def __init__(self, tipo, name, expr):
        self.tipo = tipo
        self.name = name
        self.expr = expr

    def __repr__(self):
        return f"Declaration({self.tipo}, {self.name}, {self.expr})"

class StringLit(ASTNode):
    def __init__(self, value):
        self.value = value.strip('"') # Quita las comillas
    def __repr__(self):
        return f"StringLit({self.value})"

class BoolLit(ASTNode):
    def __init__(self, value):
        self.value = True if value == 'true' else False
    def __repr__(self):
        return f"BoolLit({self.value})"

print("✅ Clases AST definidas")

✅ Clases AST definidas


### 🔹 Paso 3: Parser básico (recursivo)

In [8]:
TYPE_TOKENS = {'INT', 'FLOAT_T', 'STRING_T', 'BOOL_T'}
TYPE_MAP = {'INT': 'int', 'FLOAT_T': 'float', 'STRING_T': 'string', 'BOOL_T': 'boolean'}

class Parser:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0

    def peek(self):
        return self.tokens[self.pos] if self.pos < len(self.tokens) else (None, None)

    def match(self, expected):
        tok = self.peek()
        if tok[0] == expected:
            self.pos += 1
            return tok[1]
        raise SyntaxError(f"Esperado {expected}, pero encontró {tok}")

    def parse(self):
        return self.parse_block()

    def parse_block(self):
        statements = []
        while self.pos < len(self.tokens) and self.peek()[0] != 'RBRACE':
            statements.append(self.parse_statement())
        return Block(statements)

    def parse_statement(self):
        tok, val = self.peek()
        if tok in TYPE_TOKENS:           # NUEVO: detecta int, float, string, boolean
            return self.parse_declaration()
        elif tok == 'IF':
            return self.parse_if()
        elif tok == 'WHILE':
            return self.parse_while()
        elif tok == 'PRINT':
            return self.parse_print()
        elif tok == 'ID':
            if self.pos + 1 < len(self.tokens) and self.tokens[self.pos + 1][0] == 'ASSIGN':
                return self.parse_assign()
            else:
                raise SyntaxError(f"Sentencia inválida: variable '{val}' sin asignación")
        else:
            raise SyntaxError(f"Sentencia inválida: token inesperado '{val}'")

    def parse_declaration(self):         # NUEVO: parsea "int x = 5;"
        tok, _ = self.peek()
        tipo = TYPE_MAP[tok]
        self.pos += 1                    # consume el tipo (int, float, etc.)
        name = self.match('ID')
        self.match('ASSIGN')
        expr = self.parse_expr()
        self.match('SEMICOLON')
        return Declaration(tipo, name, expr)

    def parse_assign(self):
        name = self.match('ID')
        self.match('ASSIGN')
        expr = self.parse_expr()
        self.match('SEMICOLON')
        return Assign(name, expr)

    def parse_if(self):
        self.match('IF')
        self.match('LPAREN')
        cond = self.parse_expr()
        self.match('RPAREN')
        self.match('LBRACE')
        then_block = self.parse_block()
        self.match('RBRACE')
        else_block = None
        if self.peek()[0] == 'ELSE':
            self.match('ELSE')
            self.match('LBRACE')
            else_block = self.parse_block()
            self.match('RBRACE')
        return If(cond, then_block, else_block)

    def parse_while(self):
        self.match('WHILE')
        self.match('LPAREN')
        cond = self.parse_expr()
        self.match('RPAREN')
        self.match('LBRACE')
        body = self.parse_block()
        self.match('RBRACE')
        return While(cond, body)

    def parse_print(self):
        self.match('PRINT')
        self.match('LPAREN')
        expr = self.parse_expr()
        self.match('RPAREN')
        self.match('SEMICOLON')
        return Print(expr)

    def parse_expr(self):
        left = self.parse_addition()
        while self.peek()[0] == 'REL_OP':
            op = self.match('REL_OP')
            right = self.parse_addition()
            left = BinOp(left, op, right)
        return left

    def parse_addition(self):
        left = self.parse_multiplication()
        while self.peek()[0] == 'OP_ADD':
            op = self.match('OP_ADD')
            right = self.parse_multiplication()
            left = BinOp(left, op, right)
        return left

    def parse_multiplication(self):
        left = self.parse_factor()
        while self.peek()[0] == 'OP_MUL':
            op = self.match('OP_MUL')
            right = self.parse_factor()
            left = BinOp(left, op, right)
        return left

    def parse_factor(self):
        tok, val = self.peek()
        if tok == 'NUMBER':
            self.match('NUMBER')
            return Number(val)
        elif tok == 'STRING':            # NUEVO: "Ana"
            self.match('STRING')
            return StringLit(val)
        elif tok == 'BOOL':              # NUEVO: true / false
            self.match('BOOL')
            return BoolLit(val)
        elif tok == 'ID':
            self.match('ID')
            return Var(val)
        elif tok == 'LPAREN':
            self.match('LPAREN')
            expr = self.parse_expr()
            self.match('RPAREN')
            return expr
        else:
            raise SyntaxError(f"Token inesperado en expresión: {val}")

print("✅ Parser definido")

✅ Parser definido


### 🔹 Paso 4: Intérprete

In [9]:
class Interpreter:
    def __init__(self):
        self.env = {}    # Guarda valores
        self.types = {}  # NUEVO: Guarda los tipos declarados

    def eval(self, node):
        if isinstance(node, Block):
            for stmt in node.statements:
                self.eval(stmt)

        elif isinstance(node, Number):
            return node.value

        elif isinstance(node, Var):
            if node.name not in self.env:
                raise NameError(f"Variable '{node.name}' no está definida")
            return self.env[node.name]

        elif isinstance(node, Declaration): # NUEVO
            val = self.eval(node.expr)
            tipo_esperado = node.tipo
            if tipo_esperado == 'int' and not isinstance(val, int):
                raise TypeError(f"Error: {node.name} debe ser int")
            elif tipo_esperado == 'float' and not isinstance(val, (int, float)):
                raise TypeError(f"Error: {node.name} debe ser float")
            elif tipo_esperado == 'string' and not isinstance(val, str):
                raise TypeError(f"Error: {node.name} debe ser string")
            elif tipo_esperado == 'boolean' and not isinstance(val, bool):
                raise TypeError(f"Error: {node.name} debe ser boolean")
            self.env[node.name] = val
            self.types[node.name] = node.tipo # Guarda el tipo en la tabla
            return val

        elif isinstance(node, Assign):
            if node.var not in self.types:
                raise NameError(f"Variable '{node.var}' no declarada. Use un tipo (int, string, etc.) para crearla.")
            val = self.eval(node.expr)
            tipo_esperado = self.types[node.var]
            if tipo_esperado == 'int' and not isinstance(val, int):
                raise TypeError(f"Error: {node.var} debe ser int")
            elif tipo_esperado == 'float' and not isinstance(val, (int, float)):
                raise TypeError(f"Error: {node.var} debe ser float")
            elif tipo_esperado == 'string' and not isinstance(val, str):
                raise TypeError(f"Error: {node.var} debe ser string")
            elif tipo_esperado == 'boolean' and not isinstance(val, bool):
                raise TypeError(f"Error: {node.var} debe ser boolean")
            self.env[node.var] = val
            return val

            val = self.eval(node.expr)
            # Validación de tipos básica
            tipo_esperado = self.types[node.var]
            if tipo_esperado == 'int' and not isinstance(val, int):
                raise TypeError(f"Error de tipo: '{node.var}' es {tipo_esperado} pero recibió {type(val).__name__}")

            self.env[node.var] = val
            return val

        elif isinstance(node, BinOp):
            l = self.eval(node.left)
            r = self.eval(node.right)
            if node.op == '+': return l + r
            elif node.op == '-': return l - r
            elif node.op == '*': return l * r
            elif node.op == '/':
                if r == 0:
                    raise ZeroDivisionError("División por cero")
                return l // r
            # Operadores relacionales retornan 1/0 en lugar de True/False
            elif node.op == '==': return 1 if l == r else 0
            elif node.op == '!=': return 1 if l != r else 0
            elif node.op == '<': return 1 if l < r else 0
            elif node.op == '>': return 1 if l > r else 0
            elif node.op == '<=': return 1 if l <= r else 0
            elif node.op == '>=': return 1 if l >= r else 0

        elif isinstance(node, Print):
            print(self.eval(node.expr))

        elif isinstance(node, If):
            cond_result = self.eval(node.cond)
            # En nuestro lenguaje, 0 es falso, cualquier otro número es verdadero
            if cond_result:
                self.eval(node.then_branch)
            elif node.else_branch:
                self.eval(node.else_branch)

        elif isinstance(node, While):
            while self.eval(node.cond):
                self.eval(node.body)

        elif isinstance(node, StringLit):
            return node.value

        elif isinstance(node, BoolLit):
            return node.value

print("✅ Intérprete definido ")

✅ Intérprete definido 


### 🔹 Paso 5: Ejecutar un programa de ejemplo

In [15]:
program_code = """
int x = 5 + 3 * 2;
int y = x + 1;
if (y > 10) {
    print(y);
} else {
    print(0);
}
int z = 0;
while (z < 3) {
    print(z);
    z = z + 1;
}
"""

print("=== 🚀 EJECUTANDO PROGRAMA PRINCIPAL ===")
print("Código a ejecutar:")
print(program_code)
print("\n--- Salida del programa ---")

try:
    tokens = tokenize(program_code)
    print(f"✅ Tokenización exitosa: {len(tokens)} tokens")

    parser = Parser(tokens)
    ast = parser.parse()
    print("✅ Parsing exitoso")

    interpreter = Interpreter()
    interpreter.eval(ast)

    print(f"\n✅ Ejecución completada")
    print(f"Variables finales: {interpreter.env}")

except Exception as e:
    print(f"❌ Error: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()

=== 🚀 EJECUTANDO PROGRAMA PRINCIPAL ===
Código a ejecutar:

int x = 5 + 3 * 2;
int y = x + 1;
if (y > 10) {
    print(y);
} else {
    print(0);
}
int z = 0;
while (z < 3) {
    print(z);
    z = z + 1;
}


--- Salida del programa ---
✅ Tokenización exitosa: 61 tokens
✅ Parsing exitoso
12
0
1
2

✅ Ejecución completada
Variables finales: {'x': 11, 'y': 12, 'z': 3}


### 🧪 Programas de prueba adicionales

In [10]:
# 1. Definimos el programa como un texto (dentro de triples comillas)
codigo_test = """
int x = 10;
int y = x * 2;
print(y);
"""

# 2. Lo pasamos por nuestro sistema de procesamiento
try:
    # Convertimos el texto en tokens
    tokens = tokenize(codigo_test)

    # Creamos el árbol (AST)
    parser = Parser(tokens)
    ast = parser.parse()

    # Ejecutamos con el intérprete
    interprete = Interpreter()
    interprete.eval(ast)

    print("\n✅ Programa ejecutado correctamente.")
except Exception as e:
    print(f"❌ Error durante la ejecución: {e}")

20

✅ Programa ejecutado correctamente.


### 🔍 Herramientas de Debug

In [17]:
def debug_tokenizer(code):
    """Función para debuggear el tokenizador"""
    print(f"📝 Código: {repr(code)}")
    try:
        tokens = tokenize(code)
        print("🔍 Tokens generados:")
        for i, (tok_type, value) in enumerate(tokens):
            print(f"  {i}: {tok_type:10} = {repr(value)}")
        return tokens
    except Exception as e:
        print(f"❌ Error en tokenización: {e}")
        return None

def debug_parser(tokens):
    """Función para debuggear el parser"""
    if not tokens:
        return None
    try:
        parser = Parser(tokens)
        ast = parser.parse()
        print(f"🌳 AST generado: {ast}")
        return ast
    except Exception as e:
        print(f"❌ Error en parsing: {e}")
        return None

# Ejemplo de uso del debugger
print("=== 🔍 DEBUGGER ===")
test_code = "int x = 5; print(x);"
tokens = debug_tokenizer(test_code)
ast = debug_parser(tokens)

if ast:
    print("\n🚀 Ejecutando...")
    interpreter = Interpreter()
    interpreter.eval(ast)

=== 🔍 DEBUGGER ===
📝 Código: 'int x = 5; print(x);'
🔍 Tokens generados:
  0: INT        = 'int'
  1: ID         = 'x'
  2: ASSIGN     = '='
  3: NUMBER     = '5'
  4: SEMICOLON  = ';'
  5: PRINT      = 'print'
  6: LPAREN     = '('
  7: ID         = 'x'
  8: RPAREN     = ')'
  9: SEMICOLON  = ';'
🌳 AST generado: Block([Declaration(int, x, Number(5)), Print(Var(x))])

🚀 Ejecutando...
5


### ✅ Actividad para estudiantes

**Objetivo**: Agregar soporte para declarar variables con tipos (`int`, `float`, `string`, `boolean`) y extender el sistema para validar el tipo al momento de asignar.

#### Pasos sugeridos:
1. Agrega un nuevo nodo `Declaration(type, name, expr)` al AST.
2. Modifica el parser para soportar `int x = 5;` y `string nombre = "Ana";`
3. Añade validación de tipos en el intérprete.
4. Usa una tabla de tipos para validar las conversiones y errores.

#### Programa de prueba sugerido:
```c
int x = 10;
int y = x * 2;
print(y);
```

#### Extensiones adicionales que puedes implementar:
- **Strings**: Soporte para cadenas de texto con comillas
- **Booleanos**: Valores `true` y `false`
- **Funciones**: Definición y llamada de funciones
- **Arrays**: Soporte para listas de elementos
- **Scope**: Variables locales y globales

---

## 📋 Resumen:

1. **🔧 Tokenizador**: Reordenado las expresiones regulares para que las palabras clave se procesen antes que los identificadores
2. **🔧 Parser**: Verificación de asignaciones en `parse_statement()`
3. **🔧 Intérprete**:
   - Agregada verificación de variables no definidas
   - Manejo de división por cero
   - Operadores relacionales retornan 1/0 consistentemente
4. **🔧 Debug**: Agregadas herramientas de debugging y mensajes informativos
5. **🔧 Tests**: Programas de prueba adicionales para validar la funcionalidad

